#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In LangChain, message types (roles) are represented by different message classes. The core ones are:

The main 5 message types

- HumanMessage (role: user / human): input from the user.
- AIMessage (role: assistant / ai): the model's response; can include tool_calls.
- SystemMessage (role: system): instructions that set the model's behavior or context.
- ToolMessage (role: tool): the result of a tool/function call, linked back via tool_call_id.
- FunctionMessage (role: function): older, deprecated predecessor of ToolMessage (OpenAI's legacy function calling).

Additional types

- ChatMessage: a generic message where you set any custom role string yourself.
- RemoveMessage: a special message used in LangGraph to delete a message from the state by ID (not sent to the model).

Chunk variants

For streaming, each has a chunk version: `HumanMessageChunk, AIMessageChunk, SystemMessageChunk, ToolMessageChunk, ChatMessageChunk`, and so on.

In [2]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [3]:
model.invoke("Please tell me how to properly cook basmati rice")

AIMessage(content='**How to Cook Basmati Rice – A Step‑by‑Step Guide**\n\nBasmati rice is prized for its long, slender grains, fragrant aroma, and fluffy texture when cooked properly. Below is a reliable method that works on a stovetop and in a rice cooker. Feel free to adapt it to your taste and equipment.\n\n---\n\n## 1. Gather Your Ingredients & Equipment\n\n| Item | Typical Amount |\n|------|----------------|\n| Basmati rice | 1 cup (≈200\u202fg) |\n| Water | 1.5\u202f×\u202fthe rice volume for stovetop; 1.25\u202f×\u202ffor rice cooker (see notes) |\n| Salt | ½ tsp (optional) |\n| Oil or butter | 1 tsp (optional) |\n| Optional aromatics | 1–2 cardamom pods, a pinch of saffron, bay leaf, cinnamon stick |\n\n**Equipment**\n\n- Fine‑mesh strainer or colander\n- Medium‑sized saucepan (stovetop) or rice cooker\n- Measuring cups and spoons\n- Wooden spoon or spatula\n\n---\n\n## 2. Rinse the Rice\n\n1. **Measure** the rice into a bowl or strainer.\n2. **Rinse** under cold running water,

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [4]:
model.invoke("what is a roux")

AIMessage(content='A **roux** (pronounced “roo”) is a classic thickening base made from equal parts fat (usually butter, though oil or animal fat works too) and flour, cooked together until the flour loses its raw taste and the mixture takes on a particular color. It’s the cornerstone of many French, Cajun, and Creole sauces, soups, stews, and gravies.\n\n### How it Works\n- **Fat** coats the flour particles, preventing them from clumping.\n- **Heat** gelatinizes the starch in the flour, turning the mixture into a smooth, paste‑like thickener.\n- **Color** develops as the roux cooks, which signals the degree of flavor you’re aiming for.\n\n### Types (by color & flavor)\n\n| Color | Cooking Time | Flavor Profile | Typical Uses |\n|-------|--------------|----------------|--------------|\n| **White** | 2–3\u202fmin | Mild, almost neutral | Creamy sauces (béchamel), some soups |\n| **Blonde** | 5–7\u202fmin | Slight nutty, subtle | Velouté, lighter gravies |\n| **Brown** | 10–15\u202fmin |

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

A `Runnable` is LangChain's standard interface: any component (models, prompts, parsers, retrievers, chains) that implements the same set of methods:

- invoke / ainvoke: single input
- stream / astream: streaming output
- batch / abatch: multiple inputs

In [17]:
import json

from langchain.messages import AIMessage, HumanMessage, SystemMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem about biryani")
]

# response=model.invoke(messages)
response=await model.ainvoke(messages)
response.content

'**Biryani’s Ode**\n\nIn the hush of a kitchen, a pot whispers  \ngolden grains, each husk a tiny lantern,  \nsaffron‑kissed steam spirals like a fragrant sunrise.  \n\nCumin and cardamom break in,  \ntheir earthy pulse beating against the heat,  \nwhile cloves sigh like distant bells,  \ntossing their perfume into the rising mist.  \n\nOn the surface, a sheen of ghee,  \nmirrored in the eyes of a hungry family,  \nand the aroma swirls—  \na tapestry woven from chilies, bay leaves, and rosewater.  \n\nThe rice, a sea of pearls, rises,  \nsoft and fragrant, each grain a promise  \nof warmth, of shared stories, of home.  \n\nWhen the lid lifts, the world pauses—  \na fragrant revelation, a dance of spice and soul,  \nand the first spoonful sings  \nthe quiet triumph of patience, of love, of biryani.'

In [18]:
print(json.dumps(response.model_dump(),indent=2))

{
  "content": "**Biryani\u2019s Ode**\n\nIn the hush of a kitchen, a pot whispers  \ngolden grains, each husk a tiny lantern,  \nsaffron\u2011kissed steam spirals like a fragrant sunrise.  \n\nCumin and cardamom break in,  \ntheir earthy pulse beating against the heat,  \nwhile cloves sigh like distant bells,  \ntossing their perfume into the rising mist.  \n\nOn the surface, a sheen of ghee,  \nmirrored in the eyes of a hungry family,  \nand the aroma swirls\u2014  \na tapestry woven from chilies, bay leaves, and rosewater.  \n\nThe rice, a sea of pearls, rises,  \nsoft and fragrant, each grain a promise  \nof warmth, of shared stories, of home.  \n\nWhen the lid lifts, the world pauses\u2014  \na fragrant revelation, a dance of spice and soul,  \nand the first spoonful sings  \nthe quiet triumph of patience, of love, of biryani.",
  "additional_kwargs": {
    "reasoning_content": "We need to write a poem about biryani. It's a request. As a poetry expert, produce a creative poem. Lik

In [19]:
system_msg = SystemMessage("You are a helpful cooking assistant.")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

### Preheating a Convection Oven – Quick & Easy Guide

| Step | What to Do | Why It Matters |
|------|------------|----------------|
| **1. Check the Manual** | Look up your oven’s recommended preheat settings (most manuals say “preheat to X°” or “preheat to X° + 15 °F for convection”). | Each brand can have a tiny tweak, and you’ll avoid surprises. |
| **2. Set the Temperature** | Turn the dial or enter the number for the recipe’s temperature. | Convection ovens circulate air, so you’ll usually set **10–15 °F lower** than a conventional oven. |
| **3. Turn on the Fan** | Flip the “Fan” or “Convection” switch to “ON” (or select “Convection” mode). | The fan is what makes the air move and cooks food evenly. |
| **4. Let It Warm Up** | Close the door and let the oven run until the display shows “Ready” or the internal temperature hits the set point. | Preheating ensures your dish starts at the right temperature, giving even baking or roasting. |
| **5. Verify (Optional)** | If you’re uns

In [8]:
## Detailed info to the LLM through System message
from langchain.messages import HumanMessage, SystemMessage

system_msg = SystemMessage("""
You are a senior chef with expertise in kitchen equipment and techniques.
Always provide clear steps and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

**Preheating a convection oven**

1. **Set the temperature**  
   - Turn the dial or digital control to the temperature called for in your recipe.  
   - *Why?* The convection fan circulates air, so the oven reaches the set temperature faster and more evenly.

2. **Select the convection mode**  
   - If your oven has a “convection” button, press it.  
   - Some ovens automatically use convection when the fan is on; just make sure the fan is running.

3. **Let it preheat fully**  
   - Wait until the oven’s indicator (light or digital read‑out) shows that the set temperature has been achieved.  
   - *Why?* Opening the door before the oven is at temperature will cause a drop in heat and uneven cooking.

4. **Adjust if needed**  
   - For many recipes, reduce the stated temperature by 10–20 °F (5–10 °C) when using convection.  
   - *Why?* The circulating air cooks food more efficiently, so a lower temperature prevents over‑browning or drying.

5. **Place your dish**  
   - Put the food 

In [9]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="ravi",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [10]:
# Where msg_123 lives?
# msg_123 belongs to your HumanMessage. It stays on that object and is not copied into the response:

response = model.invoke([
  human_msg
])
# response
print(json.dumps(response.model_dump(),indent=2,ensure_ascii=False))

{
  "content": "Hello! 👋 How can I help you today?",
  "additional_kwargs": {
    "reasoning_content": "The user says \"Hello!\" The user hasn't asked a question. According to the instruction, we should respond politely, maybe ask how can we help. There's no conflict with policies. There's no disallowed content. So respond with a friendly greeting."
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 70,
      "prompt_tokens": 73,
      "total_tokens": 143,
      "completion_time": 0.084983088,
      "completion_tokens_details": {
        "reasoning_tokens": 50
      },
      "prompt_time": 0.004857514,
      "prompt_tokens_details": null,
      "queue_time": 0.050915754,
      "total_time": 0.089840602
    },
    "model_name": "openai/gpt-oss-20b",
    "system_fingerprint": "fp_c5a89987dc",
    "service_tier": "on_demand",
    "finish_reason": "stop",
    "logprobs": null,
    "model_provider": "groq"
  },
  "type": "ai",
  "name": null,
  "id": "lc_run--01a2

#### Another example

In [22]:
from langchain.messages import HumanMessage, SystemMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!, i remmember you were planning a trip to italy")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Do you remmember for which location i was planning a trip?")
]

response = model.invoke(messages)
print(response.content)

Yes—I remember you mentioned you were planning a trip to Italy. Let me know if you’d like help with anything specific (travel tips, itinerary ideas, etc.).


In [23]:
response.usage_metadata

{'input_tokens': 127,
 'output_tokens': 292,
 'total_tokens': 419,
 'output_token_details': {'reasoning': 250}}

#### Example ToolMessage

In [24]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "check_ingredient_stock",
        "args": {"ingredient": "saffron"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
stock_result = "In stock, 15g available"
tool_message = ToolMessage(
    content=stock_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("Is saffron in stock?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [25]:
tool_message

ToolMessage(content='In stock, 15g available', tool_call_id='call_123')

In [26]:
response

AIMessage(content='Yes! Saffron is in stock – we have 15\u202fg available.', additional_kwargs={'reasoning_content': 'We have to respond to user: "Is saffron in stock?" We have the function call result: saffron in stock, 15g available. We need to answer. Probably just say yes, saffron is in stock, 15g available.'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 116, 'total_tokens': 195, 'completion_time': 0.086399359, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.009631523, 'prompt_tokens_details': None, 'queue_time': 0.316662233, 'total_time': 0.096030882}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a222eb-0cd6-73d2-aaf6-77f6521f8c9e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 116, 'output_tokens': 79, 'total_tokens': 195, 'output_token_details': {'reasoning

### 🤔 Before you move on
Every response so far has been free-form text (`response.content`) that you'd have to manually parse if you wanted structured data out of it. What if you want the model to hand back a *guaranteed* schema — a Pydantic object, a TypedDict, a dataclass — instead of a string you have to pick apart yourself? That's `5-structuredoutput.ipynb`.